# Experimentación con Parámetros del Modelo: Creatividad y Control en LLMs

Este cuaderno explora cómo los parámetros de generación (temperature, top_p, presence_penalty, frequency_penalty) influyen en la creatividad, diversidad y control de las respuestas de los modelos de lenguaje. Incluye ejemplos prácticos, análisis crítico y recomendaciones para distintos casos de uso en IA aplicada a restaurantes y creatividad.

## 1. Sección de Configuración

Configuramos el cliente de Azure OpenAI usando variables de entorno y definimos la función auxiliar `get_completion`, que permite ajustar temperature, top_p, presence_penalty y frequency_penalty.

In [18]:
import os
from dotenv import load_dotenv
import openai
from typing import Optional, Dict, Any
# Load environment variables from .env file
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_API_BASE = os.getenv("OPENAI_API_BASE")
OPENAI_API_TYPE = os.getenv("OPENAI_API_TYPE", "azure")
OPENAI_API_VERSION = os.getenv("OPENAI_API_VERSION", "2023-05-15")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME")
if OPENAI_API_TYPE == 'azure':
    client = openai.AzureOpenAI(
        api_key=OPENAI_API_KEY,
        azure_endpoint=OPENAI_API_BASE,
        api_version=OPENAI_API_VERSION
    )
else:
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
def get_completion(prompt: str, temperature: float = 0, max_tokens: int = 64,
                  deployment_name: Optional[str] = None,
                  system_prompt: Optional[str] = None,
                  **extra_params) -> str:
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    response = client.chat.completions.create(
        model=deployment_name or DEPLOYMENT_NAME,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        **extra_params
    )
    return response.choices[0].message.content.strip()

## 2.1 Experimento: Temperature

El parámetro **temperature** controla el nivel de aleatoriedad en las respuestas del modelo. Un valor bajo (cercano a 0) hace que el modelo sea más determinista y predecible, mientras que valores altos (hasta 2) aumentan la creatividad y diversidad, pero también la posibilidad de respuestas menos coherentes.

In [19]:
prompt = "Write a short, catchy slogan for a new 'Truffle & Gold Burger'"
for temp in [0.0, 0.5, 1.0, 1.5]:
    print(f'--- Temperature: {temp} ---')
    print(get_completion(prompt, temperature=temp))
    print()

--- Temperature: 0.0 ---
"Indulge in Luxury: Truffle & Gold, Boldly Delicious!"

--- Temperature: 0.5 ---
"Indulge in Luxury: Truffle & Gold, Boldly Delicious!"

--- Temperature: 0.5 ---
"Indulge in Luxury: Truffle & Gold, Boldly Delicious!"

--- Temperature: 1.0 ---
"Indulge in Luxury: Truffle & Gold, Boldly Delicious!"

--- Temperature: 1.0 ---
"Indulge in Luxury: Taste the Gold, Savor the Truffle!"

--- Temperature: 1.5 ---
"Indulge in Luxury: Taste the Gold, Savor the Truffle!"

--- Temperature: 1.5 ---
"Indulge in Luxury: Truffle & Gold, Burgers Bold!"

"Indulge in Luxury: Truffle & Gold, Burgers Bold!"



**Análisis Comparativo (Temperature):**


## 2.2 Experimento: Top_p

El parámetro **top_p** (nucleus sampling) controla la probabilidad acumulada de las palabras candidatas. El modelo selecciona entre las palabras más probables hasta alcanzar la suma top_p. Valores bajos restringen la creatividad, mientras que valores altos permiten mayor diversidad.

In [20]:
prompt = "Write a short, catchy slogan for a new 'Truffle & Gold Burger'"
for top_p in [0.1, 0.5, 0.9, 1.0]:
    print(f'--- Top_p: {top_p} ---')
    print(get_completion(prompt, temperature=1.0, top_p=top_p))
    print()

--- Top_p: 0.1 ---
"Indulge in Luxury: Truffle & Gold, Boldly Delicious!"

--- Top_p: 0.5 ---
"Indulge in Luxury: Truffle & Gold, Boldly Delicious!"

--- Top_p: 0.5 ---
"Indulge in Luxury: Truffle & Gold, Boldly Delicious!"

--- Top_p: 0.9 ---
"Indulge in Luxury: Truffle & Gold, Boldly Delicious!"

--- Top_p: 0.9 ---
"Indulge in Luxury: Bite into the Gold Standard!"

--- Top_p: 1.0 ---
"Indulge in Luxury: Bite into the Gold Standard!"

--- Top_p: 1.0 ---
"Indulge in Luxury: Taste the Gold, Savor the Truffle!"

"Indulge in Luxury: Taste the Gold, Savor the Truffle!"



**Análisis Comparativo (Top_p):**


## 2.3 Experimento: Presence Penalty vs Frequency Penalty

**Presence penalty** penaliza la aparición de nuevas palabras, fomentando la diversidad temática. **Frequency penalty** penaliza la repetición de palabras ya usadas, reduciendo la redundancia. Ambos parámetros ayudan a controlar la creatividad y evitar respuestas repetitivas.

In [21]:
# Experimenta con presence_penalty y frequency_penalty
prompt = ("List 10 benefits of eating vegetables, repeating the word 'healthy' as much as possible.")
print('--- Presence Penalty: 0.0 vs 0.6 ---')
print('Penalty 0.0:')
print(get_completion(prompt, presence_penalty=0.0, max_tokens=256))
print('Penalty 0.6:')
print(get_completion(prompt, presence_penalty=0.6, max_tokens=256))
print()

print('--- Frequency Penalty: 0.0 vs 0.8 ---')
print('Penalty 0.0:')
print(get_completion(prompt, frequency_penalty=0.0, max_tokens=256))
print('Penalty 0.8:')
print(get_completion(prompt, frequency_penalty=0.8, max_tokens=256))
print()

print('--- Combination: Presence 0.6, Frequency 0.8 ---')
print(get_completion(prompt, presence_penalty=0.6, frequency_penalty=0.8, max_tokens=256))

--- Presence Penalty: 0.0 vs 0.6 ---
Penalty 0.0:
1. **Healthy** digestion: Vegetables are rich in fiber, which promotes a **healthy** digestive system.  
2. **Healthy** weight management: Low in calories and high in nutrients, vegetables support a **healthy** weight.  
3. **Healthy** skin: Vitamins and antioxidants in vegetables contribute to **healthy** and glowing skin.  
4. **Healthy** immune system: Vegetables are packed with nutrients that strengthen your immune system for a **healthy** life.  
5. **Healthy** heart: Many vegetables contain potassium and fiber, which help maintain a **healthy** heart.  
6. **Healthy** bones: Vegetables like spinach and kale are rich in calcium and vitamin K, essential for **healthy** bones.  
7. **Healthy** energy levels: Nutrient-dense vegetables provide sustained energy for a **healthy** and active lifestyle.  
8. **Healthy** brain function: Antioxidants and vitamins in vegetables support a **healthy** brain and cognitive function.  
9. **Health

**Análisis Comparativo (Penalties):**


## 2.4 Preguntas Teóricas

1. **¿Cuál es la diferencia entre top_p y temperature?**

- **Temperature**: Controla qué tan determinista o aleatoria es la respuesta del modelo. A mayor temperature, mayor creatividad y variabilidad, pero también más riesgo de respuestas menos confiables. A menor temperature, las respuestas son más consistentes y previsibles.

- **Top_p**: Limita el conjunto de opciones a las palabras (tokens) con mayor probabilidad acumulada hasta alcanzar el valor de top_p. Un top_p bajo restringe la creatividad, mientras que uno alto permite mayor variedad de respuestas.

2. **¿Por qué no conviene ajustar ambos parámetros a la vez?**

    Ajustar ambos parámetros simultáneamente puede hacer que el comportamiento del modelo sea difícil de predecir, ya que interactúan entre sí. Por ejemplo, si top_p es muy bajo, la temperatura apenas tiene efecto. Lo recomendable es modificar uno a la vez para entender su impacto y tener mayor control sobre la generación.

3. **¿En qué se diferencian presence_penalty y frequency_penalty?**

- **frequency_penalty**: Penaliza la repetición de palabras en función de cuántas veces ya han aparecido en el texto. Cuanto más se repite una palabra, menos probable es que el modelo la vuelva a usar, ayudando a evitar textos repetitivos.

- **presence_penalty**: Penaliza el uso de palabras que ya han aparecido aunque sea una sola vez. Su objetivo es empujar al modelo a introducir ideas o vocabulario nuevo, evitando que repita conceptos o términos ya mencionados.

    frequency_penalty reduce la repetición según cuántas veces aparece una palabra, mientras que presence_penalty penaliza simplemente el hecho de que ya haya aparecido una vez.

## 3. Conclusiones y Recomendaciones

A continuación se resumen las recomendaciones de configuración según el caso de uso:

| Caso de Uso                        | Temperature | Top_p | Presence Penalty | Frequency Penalty |
|-------------------------------------|-------------|-------|------------------|-------------------|
| Extracción técnica / JSON           | 0.0-0.2     | 0.1-0.3 | 0.0              | 0.0               |
| Chatbots / Atención al cliente      | 0.5         | 0.7   | 0.0-0.2          | 0.0-0.2           |
| Brainstorming creativo / Marketing  | 1.0-1.5     | 0.9-1.0 | 0.2-0.6          | 0.2-0.8           |

Ajustar estos parámetros permite adaptar el comportamiento del modelo a las necesidades específicas de cada aplicación.